# Matched experiment v2 — one victim per Kaggle account

* **Account 1:** `CELL = "std_noreg"` — baseline C (standard training, no augmentation, 35 epochs)
* **Account 2:** `CELL = "fgsm_noreg"` — FGSM victim D (FGSM-RS adversarial training, ε = 4/255, α = 0.5)

1. **Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Add the input **NIH Chest X-rays** (`nih-chest-xrays/data`).
2. Set `CELL` below, then **Save Version → Save & Run All (Commit)**.
3. If `STATUS.md` says *NOT FINISHED*, add **this notebook's latest output** as an input and Save Version again. It resumes where it stopped.

Full instructions: `Matched_Experiments_v2/kaggle/README_KAGGLE.md` in the repository.


In [ ]:
# ===================== EDIT (different CELL on each account) =====================
CELL  = "std_noreg"   # account 1: "std_noreg" (baseline C) | account 2: "fgsm_noreg" (FGSM victim D)
SEEDS = "42"          # later weeks: "42 43 44" (finished seeds are skipped)
SMOKE = False         # True = ~30-45 min pipeline test (writes results_v2_smoke/)

# FGSM account only: one victim per entry, "method:eps(/255):alpha".
# Default = recommended recipe. Optional: add a 2nd entry, e.g. "fgsm_rs:4:0.5 fgsm_rs:8:0.5",
# to train a second ε in parallel on the otherwise idle 2nd GPU (doubles the attack time).
FGSM_VARIANTS = "fgsm_rs:4:0.5"

# Baseline account only: reuse the std_noreg victim of the OLD Matched_Experiments run
# (first attach that notebook's output: Add Input -> Your Work -> Notebooks). Saves ~4 GPU hours.
IMPORT_OLD_VICTIM = False
# ==================================================================================
REPO   = "https://github.com/NCRL-IIITB/Evasion_MIA.git"
BRANCH = "main"
CODE   = "/tmp/Evasion_MIA"     # code lives outside /kaggle/working (not saved as output)


In [ ]:
# Get the code (git clone, or git pull if it is already there)
import os, subprocess

url = REPO
try:   # private repository: add a Kaggle secret called GITHUB_TOKEN (Add-ons -> Secrets)
    from kaggle_secrets import UserSecretsClient
    url = REPO.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
except Exception:
    pass

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:   # never print the URL: it may contain the token
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(url, REPO))
    return r.stdout

if os.path.isdir(os.path.join(CODE, ".git")):
    git("-C", CODE, "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, CODE)
print("Code version:", git("-C", CODE, "log", "-1", "--format=%h  %s  (%ci)"))
assert os.path.isdir(f"{CODE}/Matched_Experiments_v2"), "Matched_Experiments_v2 is not in the pushed code"


In [ ]:
# Run the pipeline: restores the previous output (if attached), trains, attacks.
# It stops by itself before Kaggle's 12-hour limit so the output is always saved.
cmd = ["python", "-u", f"{CODE}/Matched_Experiments_v2/kaggle/kaggle_pipeline.py",
       "--cell", CELL, "--seeds", *SEEDS.split()]
if CELL == "fgsm_noreg":
    cmd += ["--fgsm_variants", *FGSM_VARIANTS.split()]
if CELL == "std_noreg" and IMPORT_OLD_VICTIM:
    cmd += ["--import_old_victim"]
if SMOKE:
    cmd += ["--smoke"]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("pipeline exit code:", proc.returncode)


In [ ]:
# Status of this account (also saved as results_v2/STATUS.md in the output)
res = "/kaggle/working/" + ("results_v2_smoke" if SMOKE else "results_v2")
print(open(f"{res}/STATUS.md").read() if os.path.exists(f"{res}/STATUS.md") else "no STATUS.md yet")
